Notebook: 07_cross_dataset_validation.ipynb

Purpose: Evaluate dataset-level shift and confidence stability evidence.

Inputs:
- signal_quality_features.parquet
- twave_features.parquet
- measurement_reliability.parquet
- qtc_comparison.parquet

Outputs:
- cross_dataset_results.parquet

# 07 — Cross-Dataset Validation

Compare dataset distributions and stability across evidence domains at dataset granularity.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import entropy
sys.path.insert(0, str(Path.cwd().parent / 'src'))

root_dir = Path.cwd().parent
artifacts_dir = root_dir / 'artifacts'
paths = {
    'signal_quality': artifacts_dir / 'signal_quality_features.parquet',
    'twave': artifacts_dir / 'twave_features.parquet',
    'measurement': artifacts_dir / 'measurement_reliability.parquet',
    'qtc': artifacts_dir / 'qtc_comparison.parquet',
}
frames = {k: pd.read_parquet(p) if p.exists() else pd.DataFrame() for k, p in paths.items()}

for key, df in frames.items():
    if 'record_id' in df.columns:
        df['dataset_name'] = df['record_id'].astype(str).str.split('/', 1).str[0]
        frames[key] = df

all_datasets = set()
for df in frames.values():
    if 'dataset_name' in df.columns:
        all_datasets.update(df['dataset_name'].unique())

results = []
for dataset_name in sorted(all_datasets):
    row = {'dataset_name': dataset_name}
    lead_shift = 0.0
    morphology_shift = 0.0
    qtc_std = 0.0

    if not frames['signal_quality'].empty:
        sq = frames['signal_quality']
        group = sq[sq['dataset_name'] == dataset_name]
        if not group.empty:
            counts = group['lead_id'].value_counts(normalize=True)
            global_counts = sq['lead_id'].value_counts(normalize=True)
            lead_shift = float(entropy(counts + 1e-12, global_counts.reindex(counts.index, fill_value=1e-12).values))

    if not frames['twave'].empty:
        tw = frames['twave']
        group = tw[tw['dataset_name'] == dataset_name]
        if not group.empty:
            counts = group['morphology_cluster'].value_counts(normalize=True)
            global_counts = tw['morphology_cluster'].value_counts(normalize=True)
            morphology_shift = float(entropy(counts + 1e-12, global_counts.reindex(counts.index, fill_value=1e-12).values))

    if not frames['qtc'].empty:
        qc = frames['qtc']
        group = qc[qc['dataset_name'] == dataset_name]
        if not group.empty and 'qtc_fridericia' in group.columns:
            qtc_std = float(np.nanstd(group['qtc_fridericia'].dropna()))

    row['shift_score'] = float(np.nanmean([lead_shift, morphology_shift, qtc_std]))
    row['lead_distribution_shift'] = lead_shift
    row['morphology_shift'] = morphology_shift
    row['confidence_stability_score'] = float(1.0 / (1.0 + qtc_std)) if qtc_std >= 0 else 0.0
    results.append(row)

cross_df = pd.DataFrame(results)
expected = {'dataset_name','shift_score','lead_distribution_shift','morphology_shift','confidence_stability_score'}
assert expected.issubset(set(cross_df.columns)), 'cross dataset result schema mismatch'
assert cross_df['dataset_name'].is_unique

cross_df.to_parquet(artifacts_dir / 'cross_dataset_results.parquet', index=False)
print('Wrote cross_dataset_results.parquet')
